# MaxSim in Five Lines

The entire ColBERT scoring mechanism, matrix multiply, max, sum, implemented with random vectors and pure NumPy.

In [ ]:
import numpy as np

np.random.seed(42)

## The five lines

A query has token embeddings, one vector per token. A document has token embeddings too.
MaxSim scores a query against a document in three steps:

1. Compute the similarity between every query token and every document token.
2. For each query token, keep only the best match.
3. Sum those best matches.

That's it. That's the entire ColBERT scoring function.

In [ ]:
Q = np.random.randn(8, 128)
D = np.random.randn(20, 128)
similarity = Q @ D.T
max_per_query = similarity.max(axis=1)
score = max_per_query.sum()

print(f"Q shape:          {Q.shape}")
print(f"D shape:          {D.shape}")
print(f"Similarity shape: {similarity.shape}")
print(f"Max per query:    {max_per_query.shape} -> {max_per_query}")
print(f"Score:            {score:.4f}")

## Inspect the similarity matrix

Each row is a query token. Each column is a document token. The value at position (i, j) is the raw dot product between query token i and document token j. The starred values are the max in each row, the ones MaxSim keeps.

In [ ]:
np.random.seed(42)
Q = np.random.randn(8, 128)
D = np.random.randn(20, 128)
similarity = Q @ D.T

print(f"Similarity matrix ({similarity.shape[0]} query tokens x {similarity.shape[1]} doc tokens):\n")

max_indices = similarity.argmax(axis=1)

for i in range(similarity.shape[0]):
    row_strs = []
    for j in range(similarity.shape[1]):
        val = similarity[i, j]
        if j == max_indices[i]:
            row_strs.append(f"[{val:6.2f}]")
        else:
            row_strs.append(f" {val:6.2f} ")
    print(f"  q{i}: " + "".join(row_strs))

print(f"\nMax per row: {similarity.max(axis=1)}")
print(f"Sum of maxes (MaxSim score): {similarity.max(axis=1).sum():.4f}")

## Compare two documents

If MaxSim is a relevance score, different documents should get different scores against the same query. Here, both documents are random noise so neither is meaningfully "relevant", but the scores will differ because the random vectors are different.

In [ ]:
np.random.seed(42)
Q = np.random.randn(8, 128)
D1 = np.random.randn(20, 128)
D2 = np.random.randn(20, 128)

sim1 = Q @ D1.T
sim2 = Q @ D2.T

score1 = sim1.max(axis=1).sum()
score2 = sim2.max(axis=1).sum()

print(f"Q shape:  {Q.shape}")
print(f"D1 shape: {D1.shape}")
print(f"D2 shape: {D2.shape}")
print()
print(f"Score(Q, D1) = {score1:.4f}")
print(f"Score(Q, D2) = {score2:.4f}")
print()
if score1 > score2:
    print(f"D1 wins by {score1 - score2:.4f}")
else:
    print(f"D2 wins by {score2 - score1:.4f}")

## L2 normalization

The raw dot products above are unbounded, they can be any real number. After L2-normalizing each vector to unit length, dot products become cosine similarities, bounded to [-1, 1]. This is what ColBERT actually does: the projection layer's output is always L2-normalized before scoring.

In [ ]:
np.random.seed(42)
Q = np.random.randn(8, 128)
D = np.random.randn(20, 128)

Q_norm = Q / np.linalg.norm(Q, axis=1, keepdims=True)
D_norm = D / np.linalg.norm(D, axis=1, keepdims=True)

print(f"Before normalization — norm of Q[0]: {np.linalg.norm(Q[0]):.4f}")
print(f"After normalization  — norm of Q_norm[0]: {np.linalg.norm(Q_norm[0]):.4f}")
print()

sim_raw = Q @ D.T
sim_norm = Q_norm @ D_norm.T

print(f"Raw dot products     — min: {sim_raw.min():.4f}, max: {sim_raw.max():.4f}")
print(f"Cosine similarities  — min: {sim_norm.min():.4f}, max: {sim_norm.max():.4f}")
print()

score_raw = sim_raw.max(axis=1).sum()
score_norm = sim_norm.max(axis=1).sum()

print(f"MaxSim (raw):        {score_raw:.4f}")
print(f"MaxSim (normalized): {score_norm:.4f}")

## The key insight

The interaction happens at **token granularity**. Every query token gets to "look" at every document token and pick its best match. This is fundamentally different from comparing two single vectors.

A bi-encoder compresses all token information into one vector per text, then compares those two vectors. That compression is lossy, it throws away which-token-matched-which information.

MaxSim avoids the compression entirely. The cost is a larger similarity computation (a matrix instead of a single dot product), but the reward is that compositional meaning survives.

## Verify with the library function

The `colbert_from_scratch` package wraps the same logic into a reusable function. It normalizes by default (matching ColBERT's actual behavior), so the result should match the normalized score above.

In [ ]:
import sys
sys.path.insert(0, "..")

from colbert_from_scratch.maxsim import maxsim_np

np.random.seed(42)
Q = np.random.randn(8, 128)
D = np.random.randn(20, 128)

library_score = maxsim_np(Q, D, normalize=True)

Q_norm = Q / np.linalg.norm(Q, axis=1, keepdims=True)
D_norm = D / np.linalg.norm(D, axis=1, keepdims=True)
manual_score = (Q_norm @ D_norm.T).max(axis=1).sum()

print(f"Library score: {library_score:.4f}")
print(f"Manual score:  {manual_score:.4f}")
print(f"Match: {np.isclose(library_score, manual_score)}")